# Model.ipynb

In [2]:
import numpy as np
import pandas as pd
import sklearn

In [3]:
print(np.__version__, pd.__version__, sklearn.__version__)

2.5.2 3.0.5 1.9.0


In [4]:
df = pd.read_csv("../data/penguins.csv")
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


## Model

In [12]:
import copy
import numpy as np

def sigmoid(z):
    '''
    Sigmoid function f(z) = 1 / (1 + exp(-z)) (same as HW4).
    @params:
        z: a scalar or Numpy array
    @return:
        sigmoid function evaluated at x (applied element-wise if it is an array)
    '''
    return 1.0 / (1.0 + np.exp(-z))

class BinaryLogisticRegression:
    '''
    L2-regularized binary logistic regression trained with gradient descent (based on HW4).

    Minimizes L_S(w) = (1/m) sum_i log(1 + exp(-y_i <w, x_i>)) + lambda * ||w||^2,
    with lambda = 1 / (2 * C * m). The last column of X is 1 (bias), and the bias is not regularized.
    '''

    def __init__(self, C=1.0, alpha=None, num_epochs=100000, conv_threshold=1e-8):
        '''
        Initializes a BinaryLogisticRegression classifier.
        @attrs:
            C: inverse regularization strength (same meaning as in sklearn), must be > 0
            alpha: the learning rate; if None, alpha = 1 / L, where L is an upper bound on the
                   curvature of the loss, which guarantees that every step decreases the loss
            num_epochs: the maximum number of epochs (gradient descent steps)
            conv_threshold: stop when max |gradient| < conv_threshold
            weights: the weights of the model, a (1, d) Numpy array (last entry is the bias)
            loss_history: the training loss after every epoch
        '''
        if C <= 0:
            raise ValueError("C must be positive.")
        if alpha is not None and alpha <= 0:
            raise ValueError("alpha must be positive.")
        self.C = C
        self.alpha = alpha
        self.num_epochs = num_epochs
        self.conv_threshold = conv_threshold
        self.weights = None
        self.loss_history = []

    def _check_labels(self, Y):
        '''
        Raises a ValueError unless every label is 0 or 1.
        @params:
            Y: a 1D Numpy array of labels
        @return:
            None
        '''
        if not np.all(np.isin(Y, [0, 1])):
            raise ValueError("Binary labels must be 0 or 1.")

    def _lambda(self, m):
        '''
        Regularization strength lambda = 1 / (2 * C * m) for a training set of m examples.
        @params:
            m: the number of training examples
        @return:
            a float, lambda
        '''
        return 1.0 / (2.0 * self.C * m)

    def _gradient(self, X, Y):
        '''
        Computes the gradient of the regularized loss (same formula as HW4):
        dL/dw_j = (1/m) sum_i (h(x_i) - y_i) x_ij + 2 * lambda * w_j, with no regularization
        term for the bias (the last weight).
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels (0 or 1) for each example
        @return:
            a (1, d) Numpy array, the gradient of the loss with respect to the weights
        '''
        m = X.shape[0]
        prob = sigmoid(X @ self.weights.T).flatten()
        gradient = np.mean((prob - Y)[:, None] * X, axis=0)
        reg = 2 * self._lambda(m) * self.weights.flatten()
        reg[-1] = 0.0  #do not regularize the bias
        return (gradient + reg).reshape(1, -1)

    def train(self, X, Y):
        '''
        Trains the model using gradient descent.
        Pseudo-code:
            w <- 0
            for epoch = 1, ..., num_epochs:
                g <- gradient of the loss on the whole training set
                if max |g| < conv_threshold: stop
                w <- w - alpha * g
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels (0 or 1) for each example
        @return:
            num_epochs: an integer, the number of epochs taken to reach convergence
        '''
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y, dtype=float)
        self._check_labels(Y)
        m, d = X.shape
        self.weights = np.zeros((1, d))
        self.loss_history = []

        alpha = self.alpha
        if alpha is None:
            # The Hessian of the log loss is at most (1/4) X^T X / m, and the regularizer adds
            # 2 * lambda, so L below bounds the curvature and alpha = 1 / L is always safe.
            L = 0.25 * np.linalg.norm(X, 2) ** 2 / m + 2 * self._lambda(m)
            alpha = 1.0 / L

        for epoch in range(self.num_epochs):
            gradient = self._gradient(X, Y)
            if np.max(np.abs(gradient)) < self.conv_threshold:
                return epoch
            self.weights = self.weights - alpha * gradient
            self.loss_history.append(self.loss(X, Y))
        return self.num_epochs

    def decision_function(self, X):
        '''
        Computes the score <w, x> of each example (positive -> label 1).
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array with one score for each row in X
        '''
        if self.weights is None:
            raise RuntimeError("Model is not trained yet; call train() first.")
        return (np.asarray(X, dtype=float) @ self.weights.T).flatten()

    def predict_proba(self, X):
        '''
        Computes the probability that each example has label 1.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array of probabilities h(x) = sigmoid(<w, x>)
        '''
        return sigmoid(self.decision_function(X))

    def predict(self, X):
        '''
        Computes predictions based on the learned weights and examples X.
        A probability of exactly 0.5 (score 0) is predicted as 0, the same as sklearn.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array with one predicted label (0 or 1) for each row in X
        '''
        return (self.decision_function(X) > 0).astype(int)

    def loss(self, X, Y):
        '''
        The log loss is computed as log(1 + exp(-y * <w, x>)) from Lecture 5,
        where the labels are converted from {0, 1} to y in {-1, 1}.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels (0 or 1) for each example
        @return:
            a float, the loss of the model on the dataset
        '''
        Y = np.asarray(Y, dtype=float)
        self._check_labels(Y)
        z = self.decision_function(X)
        y = 2 * Y - 1 #change from {0,1} to {-1,1}
        log_loss = np.mean(np.log(1 + np.exp(-y * z)))
        w = self.weights.flatten()[:-1]
        return float(log_loss + self._lambda(len(Y)) * w @ w)

    def accuracy(self, X, Y):
        '''
        Outputs the accuracy of the trained model on a given dataset X and labels Y.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            a float number indicating accuracy (between 0 and 1)
        '''
        return float(np.mean(self.predict(X) == Y))

class OneVsAll(object):
    '''
    One-versus-All multiclass classifier (Lecture 5; textbook Section 17.1).

    Training: for each class i, relabel the examples as 1 if their label is i and 0 otherwise,
    and train one binary classifier h_i.
    Prediction: h(x) = argmax_i h_i(x), using the scores of the binary classifiers.
    Ties go to the first class (np.argmax), the same as sklearn's OneVsRestClassifier.
    '''
    def __init__(self, base_model):
        '''
        Initializes a OneVsAll classifier.
        @attrs:
            base_model: an untrained binary classifier with train(X, Y), Y in {0, 1}, and
                        decision_function(X); a copy of it is trained for every class
            classes: the sorted class labels, a 1D Numpy array of length k
            models: the k trained binary classifiers
        '''
        self.base_model = base_model
        self.classes = None
        self.models = []

    def train(self, X, Y):
        '''
        Trains one binary classifier per class.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels (any type, e.g. strings)
        @return:
            None
        '''
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y)
        self.classes = np.unique(Y)  # sorted, the same order as sklearn's classes_
        if len(self.classes) < 2:
            raise ValueError("Need at least 2 classes to train a multiclass model.")
        self.models = []
        for c in self.classes:
            # Lecture 5: S_i = (x_t, (-1)^1[y_t != i]), i.e. +1 if y_t = i and -1 otherwise.
            # Our binary learner (HW4 logistic regression) uses labels {0, 1}, so we use 1 / 0.
            Y_binary = (Y == c).astype(int)
            model = copy.deepcopy(self.base_model)
            model.train(X, Y_binary)
            self.models.append(model)

    def decision_function(self, X):
        '''
        Computes the score of every binary classifier for each example.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 2D Numpy array of shape (n_examples, k); column i holds h_i(x)
        '''
        if not self.models:
            raise RuntimeError("Model is not trained yet; call train() first.")
        return np.column_stack([model.decision_function(X) for model in self.models])

    def predict(self, X):
        '''
        Computes predictions based on the trained binary classifiers.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array with one predicted class label for each row in X
        '''
        return self.classes[np.argmax(self.decision_function(X), axis=1)]

    def loss(self, X, Y):
        '''
        Returns the average 0-1 loss (the fraction of misclassified examples).
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            a float, the average 0-1 loss
        '''
        return float(np.mean(self.predict(X) != np.asarray(Y)))

    def accuracy(self, X, Y):
        '''
        Outputs the accuracy of the trained model on a given dataset X and labels Y.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            a float number indicating accuracy (between 0 and 1)
        '''
        return 1 - self.loss(X, Y)

class AllPairs(object):
    '''
    All-Pairs multiclass classifier (Lecture 5; textbook Section 17.1).

    Training: for every pair of classes i < j, keep only the examples of those two classes and
    train one binary classifier h_ij. Following sklearn's OneVsOneClassifier, class i is labeled 0
    and class j is labeled 1 (the lecture uses +1 for i and -1 for j; the votes are the same).
    Prediction: every h_ij votes for i or j, and the class with the most votes wins. Ties are
    broken by the summed scores, squashed by x / (3(|x| + 1)) into (-1/3, 1/3) so that they can
    never overturn a difference of one vote (exactly as in sklearn).
    '''
    def __init__(self, base_model):
        '''
        Initializes an AllPairs classifier.
        @attrs:
            base_model: an untrained binary classifier with train(X, Y), Y in {0, 1}, predict(X)
                        and decision_function(X); a copy of it is trained for every pair of classes
            classes: the sorted class labels, a 1D Numpy array of length k
            models: the k(k-1)/2 trained binary classifiers
            pairs: the (i, j) class indices of each binary classifier
        '''
        self.base_model = base_model
        self.classes = None
        self.models = []
        self.pairs = []

    def train(self, X, Y):
        '''
        Trains one binary classifier per pair of classes.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels (any type, e.g. strings)
        @return:
            None
        '''
        X = np.asarray(X, dtype=float)
        Y = np.asarray(Y)
        self.classes = np.unique(Y)
        k = len(self.classes)
        if k < 2:
            raise ValueError("Need at least 2 classes to train a multiclass model.")
        self.models, self.pairs = [], []
        for i in range(k):
            for j in range(i + 1, k):
                # Lecture 5: keep only the examples with y_t = i or y_t = j
                mask = (Y == self.classes[i]) | (Y == self.classes[j])
                # Lecture 5 labels class i as +1 and class j as -1; we follow sklearn's
                # OneVsOneClassifier (class i -> 0, class j -> 1). The votes are the same.
                Y_binary = (Y[mask] == self.classes[j]).astype(int)
                model = copy.deepcopy(self.base_model)
                model.train(X[mask], Y_binary)
                self.models.append(model)
                self.pairs.append((i, j))

    def decision_function(self, X):
        '''
        Computes the votes plus the tie-breaking confidence of every class.
        votes[:, c] = number of binary classifiers that predict class c
        conf[:, c]  = sum of the scores in favor of class c
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 2D Numpy array of shape (n_examples, k), votes + conf / (3 * (|conf| + 1))
        '''
        if not self.models:
            raise RuntimeError("Model is not trained yet; call train() first.")
        X = np.asarray(X, dtype=float)
        n, k = X.shape[0], len(self.classes)
        votes = np.zeros((n, k))
        conf = np.zeros((n, k))
        for (i, j), model in zip(self.pairs, self.models):
            score = model.decision_function(X)  # > 0 favors class j, < 0 favors class i
            prediction = model.predict(X)
            conf[:, i] -= score
            conf[:, j] += score
            votes[prediction == 0, i] += 1
            votes[prediction == 1, j] += 1
        return votes + conf / (3 * (np.abs(conf) + 1))

    def predict(self, X):
        '''
        Computes predictions by majority vote of the binary classifiers.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array with one predicted class label for each row in X
        '''
        return self.classes[np.argmax(self.decision_function(X), axis=1)]

    def loss(self, X, Y):
        '''
        Returns the average 0-1 loss (the fraction of misclassified examples).
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            a float, the average 0-1 loss
        '''
        return float(np.mean(self.predict(X) != np.asarray(Y)))

    def accuracy(self, X, Y):
        '''
        Outputs the accuracy of the trained model on a given dataset X and labels Y.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            a float number indicating accuracy (between 0 and 1)
        '''
        return 1 - self.loss(X, Y)

## Check Model